## NAKO Dataset

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import glob
from itertools import cycle
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from utils import nako
from utils import helpers

pd.options.display.float_format = "{:,.2f}".format
colors = sns.color_palette('pastel', 10)
colors = cycle([mpl.colors.rgb2hex(c) for c in colors])

In [ ]:
image_types_ab = ['rt_leftcent', 'rt_leftnasal', 'rt_rightcent', 'rt_rightnasal']
image_types_labels = ['Left Central', 'Left Nasal', 'Right Central', 'Right Nasal']
n_participants = 48460

nako_paths = nako.get_nako_paths('590', image_dir='images')

# List of corrupted and ungradable images
corrupted_images, lowres_images, ungradable_images = nako.get_unusable_images(nako_paths['quality_dir'])

### Fundus Images
From 2 different retinal fields: nasal and central

In [ ]:
image_idx = np.random.randint(100000, 148462)

fig, ax = plt.subplots(1, 4, figsize=(14, 4))

for i, image_type in enumerate(nako.IMAGE_TYPES):
    img = helpers.load_image(os.path.join(nako_paths['image_dir'], f'{image_type}_1', f'{image_idx}.jpg'))
    ax[i].imshow(img)
    ax[i].title.set_text(image_types_labels[i])
    ax[i].axes.get_xaxis().set_visible(False)
    ax[i].axes.get_yaxis().set_visible(False)

plt.tight_layout()
# fig.savefig('plots/fundus_fields.svg')
# Access participant's metadata: df.loc[image_idx]['basis_age']

In [ ]:
# Number of fundus images per type
df_images = pd.DataFrame(columns=['Type', 'Index', 'All', 'Usable', 'Gradable'])
for i, image_type in enumerate(nako.IMAGE_TYPES):
    for j in range(1, 4):
        image_dir = os.path.join(nako_paths['image_dir'], image_type + '_' + str(j))
        all_images = glob.glob(os.path.join(image_dir, '*.jpg'))
        all_images = ['/'.join(p.split(sep='/')[-2:]) for p in all_images]

        # Filter out corrupted images
        usable_images = list(set(all_images).difference(set(corrupted_images)))

        # Filter out ungradable images
        gradable_images = list(set(usable_images).difference(set(ungradable_images)))

        df_images_append = pd.DataFrame([{'Type':image_type, 'Index': j, 'All': len(all_images), 'Usable': len(usable_images), 'Gradable': len(gradable_images)}])
        df_images = pd.concat([df_images, df_images_append], ignore_index=True)

df_images['All %'] = df_images['All'] * 100 / n_participants   
df_images['Usable %'] = df_images['Usable'] * 100 / n_participants
df_images['Gradable %'] = df_images['Gradable'] * 100 / n_participants             

df_images

In [ ]:
# Overlap between folders
for i, image_type in enumerate(nako.IMAGE_TYPES):
    ids_list, folder_size = [], []
    for j in range(1, 4):
        image_dir = os.path.join(nako_paths['image_dir'], image_type + '_' + str(j))
        image_paths = glob.glob(os.path.join(image_dir, '*.jpg'))
        image_paths = ['/'.join(p.split(sep='/')[-2:]) for p in image_paths]

        # Filter out corrupted and ungradable images
        # image_paths = list(set(image_paths).difference(set(corrupted_images + ungradable_images)))
        image_ids = [int(os.path.basename(p).split(sep='.')[0]) for p in image_paths]
        image_ids.sort()
        ids_list.append(image_ids)
        folder_size.append(len(image_ids))

    # Overlap of images in each folder
    overlap1 = len(set(ids_list[1]).difference(set(ids_list[0])))
    overlap2 = len(set(ids_list[2]).difference(set(ids_list[0])))
    overlap3 = len(set(ids_list[2]).difference(set(ids_list[1])))

    print(f'Overlap for {image_type}:')
    print(f'Images from folder 2 not present in folder 1: {overlap1} out of {folder_size[1]}.')
    print(f'Images from folder 3 not present in folder 1: {overlap2} out of {folder_size[2]}.')
    print(f'Images from folder 3 not present in folder 2: {overlap3} out of {folder_size[2]}.')
    print('--------------------------------------------------------')

### Features / Labels

The dataset holds for 48.460 records with 547 features.

In [ ]:
date_conv = lambda x: pd.to_datetime(x, format='%Y-%m-%d')
date_cols = ['{}_{}_date'.format(image_type, j) for image_type in image_types_ab for j in range(1,4)]

df_labels = pd.read_csv(nako_paths['labels_file'], index_col='ID', converters=dict.fromkeys(date_cols, date_conv))
print('Size of the dataset: ', df_labels.shape)

decoder = nako.FeatureDecoder(nako_paths['metadata_file'])

age_bins = list(range(20, 80, 5))
age_bins[0] = 19

In [ ]:
df_labels.head()

#### Description of demographics, blood pressure, visual acuity, eye diseases

In [ ]:
feature_names = ['basis_age', 'a_rr_sys', 'a_rr_dia', 'vi_l_log_mar_einstf', 'vi_r_log_mar_einstf']
df = df_labels[feature_names].copy()

for feature_name in feature_names:
    mapping, feature_label = decoder.decode(feature_name)
    df, mapping = decoder.group_nans(feature_name, mapping, df)

    if 'log_mar' in feature_name:
        df[feature_name] = df[feature_name].replace(7.70, np.nan)

df.describe()

In [ ]:
feature_names = ['basis_sex', 'a_ethn1', 'basis_uort', 'a_smok_stat_qn', 'd_an_aug_1', 'd_an_aug_2', 'd_an_aug_3', 'd_an_metdm2c_2', 'd_an_metdm2c_1','d_an_met_1', 'a_rr_hypertens']
df = df_labels[feature_names].copy()

for feature_name in feature_names:
    mapping, feature_label = decoder.decode(feature_name)
    df, mapping = decoder.group_nans(feature_name, mapping, df)
    count = df[feature_name].map(mapping).value_counts(normalize=True)
    count = [f'{index}: {100*value:.2f}%' for index, value in count.items()]
    
    print(feature_label)
    print(', '.join(count))

In [ ]:
mapping, label = decoder.decode('basis_sex')
crosstab = pd.crosstab(df_labels['basis_ageclass'],  df_labels['basis_sex'], rownames=['Age range (years)'], colnames=['Sex'], margins=True, margins_name='Total')
row_labels = {i : f'{i} - {i+9}' for i in range(10, 80, 10)} | {'Total': 'All ages'}
crosstab.index = crosstab.index.map(row_labels)
col_labels = {key: f'{val}, n (%)' for key, val in mapping.items()} | {'Total': 'Total, n (%)'}
crosstab.columns = crosstab.columns.map(col_labels)
crosstab = crosstab.map(lambda x: f'{x} ({100*x/df_labels.shape[0]:.2f}%)')
crosstab

#### Demographics

In [ ]:
fig, ax = plt.subplots(1,4, figsize=(12, 4))
ax = ax.flatten()

feature_names = ['basis_age', 'basis_sex', 'a_ethn1', 'basis_uort']
df = df_labels[feature_names].copy()

# Age
feature_name = 'basis_age'
mapping, label = decoder.decode(feature_name)
sns.histplot(x=feature_name, data=df, bins=age_bins, stat='percent', shrink=0.7, color=next(colors), ax=ax[0])
ax[0].set_xticks(list(range(20, 80, 10)))
ax[0].set_ylabel('Percentage of participants')
ax[0].set_xlabel('Age (5 year bins)')
ax[0].set_title(label)

for i, feature_name in enumerate(feature_names[1:], start=1):
    mapping, label = decoder.decode(feature_name)
    df, mapping = decoder.group_nans(feature_name, mapping, df)
    mapping = decoder.wrap_mapping(mapping, 18)
    df[feature_name] = df[feature_name].map(mapping)
    df = nako.sort_categorical_df(df, feature_name)
    sns.histplot(x=feature_name, data=df, stat='percent', shrink=0.7, color=next(colors), ax=ax[i])
    # ax[i].bar_label(ax[i].containers[0], fmt='%.1f')
    ax[i].set_xlabel('')
    ax[i].set_ylabel('')
    ax[i].set_title(label)

ax[2].tick_params(axis='x', rotation=90, labelsize=9)
ax[3].tick_params(axis='x', rotation=90, labelsize=8)

plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(6, 4))

feature_names = ['basis_age', 'basis_sex']
df = df_labels[feature_names].copy()

feature_name = feature_names[1]
mapping, label = decoder.decode(feature_name)
df, mapping = decoder.group_nans(feature_name, mapping, df)
df[feature_name] = df[feature_name].map(mapping)
df = nako.sort_categorical_df(df, feature_name)

age_bins = list(range(20, 80, 5))
age_bins[0] = 19

mapping, label = decoder.decode(feature_name)
sns.histplot(x='basis_age', data=df, bins=age_bins, stat='percent', shrink=0.7, multiple='dodge', color='lightgray', ec=None, ax=ax)
sns.histplot(x='basis_age', data=df, hue='basis_sex', bins=age_bins, stat='percent', shrink=0.7, multiple='dodge',ec=None, ax=ax)
sns.move_legend(ax, loc='upper right', title=None)
ax.set_xticks(list(range(20, 80, 10)))
ax.set_ylabel('Percentage of participants')
ax.set_xlabel('Age (5 year bins)')
ax.set_title('Age Distribution')
plt.tight_layout()

### Age distribution per site

In [ ]:
feature_names = ['basis_age', 'basis_uort']
df = df_labels[feature_names].copy()

feature_name = feature_names[1]
mapping, label = decoder.decode(feature_name)
df, mapping = decoder.group_nans(feature_name, mapping, df)
df[feature_name] = df[feature_name].map(mapping)
df = nako.sort_categorical_df(df, feature_name)

fig, ax = plt.subplots(1,1, figsize=(7, 5))
sns.kdeplot(x=feature_names[0], hue=feature_names[1], data=df, common_norm=False, ax=ax)
box = ax.get_position()
ax.set_position([box.x0, box.y0, box.width * 0.8, box.height])
sns.move_legend(ax, 'center left', bbox_to_anchor=(1,0.5), title=None)
ax.set_xlabel('Age (years)')
ax.set_title('Age distribution by site (kde)')
plt.tight_layout()

In [ ]:
# Gender distribution by site
fig, ax = plt.subplots(1,1, figsize=(6, 4))

feature_names = ['basis_sex', 'basis_uort']
df = df_labels[feature_names].copy()

for feature_name in feature_names:
    mapping, label = decoder.decode(feature_name)
    df, mapping = decoder.group_nans(feature_name, mapping, df)
    df[feature_name] = df[feature_name].map(mapping)
    df = nako.sort_categorical_df(df, feature_name)


mapping, label = decoder.decode(feature_name)
sns.histplot(x='basis_uort', data=df, stat='percent', shrink=0.7, multiple='dodge', color='lightgray', ec=None, ax=ax)
sns.histplot(x='basis_uort', data=df, hue='basis_sex', stat='percent', shrink=0.7, multiple='dodge',ec=None, ax=ax)
sns.move_legend(ax, loc='upper right', title=None)
ax.set_ylabel('Percentage of participants')
ax.set_xlabel('Site')
ax.set_title('Gender distribution by site')
ax.tick_params(axis='x', rotation=90, labelsize=8)
plt.tight_layout()

### Smoking

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(10, 4))
feature_names = ['a_smok_stat_qn', 'a_smok_stat_noint']
df = df_labels[feature_names].copy()

for i, feature_name in enumerate(feature_names):
    mapping, label = decoder.decode(feature_name)
    df, mapping = decoder.group_nans(feature_name, mapping, df)
    mapping = decoder.wrap_mapping(mapping, width=16)
    df[feature_name] = df[feature_name].map(mapping)
    sns.histplot(x=feature_name, data=df, stat='percent', shrink=0.7, color=next(colors), ax=ax[i])
    ax[i].bar_label(ax[i].containers[0], fmt='%.3f')
    ax[i].set_xlabel('')
    ax[i].set_title(label)
    
ax[0].set_ylabel('Percentage of participants')
plt.tight_layout()

### Blood pressure

In [ ]:
feature_names = ['a_rr_sys', 'a_rr_dia', 'a_rr_hypertens']
df = df_labels[feature_names].copy()

fig, ax = plt.subplots(1,3, figsize=(12, 4))

for i, feature_name in enumerate(feature_names[:-1]):
    mapping, label = decoder.decode(feature_name)
    df, mapping = decoder.group_nans(feature_name, mapping, df)
    sns.histplot(x=feature_name, data=df, bins=20, stat='percent', shrink=0.7, color=next(colors), ax=ax[i])
    ax[i].set_title(label.capitalize())
    ax[i].set_xlabel('')
    ax[i].set_ylabel('')

mapping, label = decoder.decode(feature_names[2])
df, mapping = decoder.group_nans(feature_names[2], mapping, df)
df[feature_names[2]] = df[feature_names[2]].map(mapping)
df = nako.sort_categorical_df(df, feature_names[2])
sns.histplot(x=feature_names[2], data=df, stat='percent', shrink=0.7, color=next(colors), ax=ax[2])
ax[2].bar_label(ax[2].containers[0], fmt=lambda x: f'{x:.3f}' if x > 0 else '')
ax[2].set_title(label.capitalize())
ax[2].set_xlabel('')
ax[2].set_ylabel('')
    
ax[0].set_ylabel('Percentage of participants')
plt.tight_layout()

### Eye Diseases + Diabetes

In [ ]:
feature_names = ['d_an_aug_1', 'd_an_aug_2', 'd_an_aug_3', 'd_an_metdm2c_2', 'd_an_metdm2c_1','d_an_met_1']
df = df_labels[feature_names].copy()

fig, ax = plt.subplots(1, 6, figsize=(21, 4))

for i, feature_name in enumerate(feature_names):
    mapping, label = decoder.decode(feature_name)
    df, mapping = decoder.group_nans(feature_name, mapping, df)
    df[feature_name] = df[feature_name].map(mapping)
    df = nako.sort_categorical_df(df, feature_name) 
    sns.histplot(x=feature_name, data=df, bins=20, stat='percent', shrink=0.7, color=next(colors), ax=ax[i])
    ax[i].set_title(label.capitalize())
    ax[i].bar_label(ax[i].containers[0], fmt='%.3f')
    ax[i].set_xlabel('')
    ax[i].set_ylabel('')
    
ax[0].set_ylabel('Percentage of participants')
plt.tight_layout()
    

In [ ]:
feature_names = ['basis_age', 'd_an_aug_1', 'd_an_aug_2', 'd_an_aug_3', 'd_an_metdm2c_2', 'd_an_metdm2c_1','d_an_met_1']
df = df_labels[feature_names].copy()

fig, ax = plt.subplots(1, 6, figsize=(21, 4))

for i, feature_name in enumerate(feature_names[1:]):
    mapping, label = decoder.decode(feature_name)
    df, mapping = decoder.group_nans(feature_name, mapping, df)
    df[feature_name] = df[feature_name].map(mapping)

    df_filtered = df[df[feature_name] == 'Yes']
    sns.histplot(x='basis_age', data=df_filtered, bins=age_bins, stat='percent', shrink=0.7, color=next(colors), ax=ax[i])

    ax[i].set_xticks(list(range(20, 80, 10)))
    ax[i].set_title(label.capitalize())
    ax[i].set_xlabel('Age (5 year bins)')
    ax[i].set_ylabel('')
    
ax[0].set_ylabel('Percentage of diseased participants')
fig.suptitle('Age distribution for eye diseases')
plt.tight_layout()

In [ ]:
feature_names = ['basis_sex', 'd_an_aug_1', 'd_an_aug_2', 'd_an_aug_3', 'd_an_metdm2c_2', 'd_an_metdm2c_1','d_an_met_1']
df = df_labels[feature_names].copy()

feature_name = feature_names[0]
mapping, label = decoder.decode(feature_name)
df, mapping = decoder.group_nans(feature_name, mapping, df)
df[feature_name] = df[feature_name].map(mapping)

fig, ax = plt.subplots(1, 6, figsize=(21, 4))

for i, feature_name in enumerate(feature_names[1:]):
    mapping, label = decoder.decode(feature_name)
    df, mapping = decoder.group_nans(feature_name, mapping, df)
    df[feature_name] = df[feature_name].map(mapping)

    df_filtered = df[df[feature_name] == 'Yes']
    sns.histplot(x='basis_sex', data=df_filtered, stat='percent', shrink=0.7, color=next(colors), ax=ax[i])

    ax[i].bar_label(ax[i].containers[0], fmt='%.3f')
    ax[i].set_title(label.capitalize())
    ax[i].set_xlabel('')
    ax[i].set_ylabel('')
    
ax[0].set_ylabel('Percentage of diseased participants')
fig.suptitle('Gender distribution for eye diseases')
plt.tight_layout()

In [ ]:
feature_names = ['basis_age', 'basis_sex', 'd_an_aug_1', 'd_an_aug_2', 'd_an_aug_3', 'd_an_metdm2c_1','d_an_met_1']
df = df_labels[feature_names].copy()

feature_name = feature_names[1]
mapping, label = decoder.decode(feature_name)
df, mapping = decoder.group_nans(feature_name, mapping, df)
df[feature_name] = df[feature_name].map(mapping)
df = nako.sort_categorical_df(df, feature_name)

fig, ax = plt.subplots(1, 5, figsize=(18, 4), sharey=True)

for i, feature_name in enumerate(feature_names[2:]):
    mapping, label = decoder.decode(feature_name)
    df, mapping = decoder.group_nans(feature_name, mapping, df)
    df[feature_name] = df[feature_name].map(mapping)

    df_filtered = df[df[feature_name] == 'Yes']
    sns.histplot(x='basis_age', data=df_filtered, bins=age_bins, stat='percent', shrink=0.7, multiple='dodge', color='lightgray', ec=None, ax=ax[i])
    sns.histplot(x='basis_age', data=df_filtered, hue='basis_sex', bins=age_bins, stat='percent', shrink=0.7, multiple='dodge',ec=None, color=next(colors), ax=ax[i])

    ax[i].set_xticks(list(range(20, 80, 10)))
    ax[i].set_title(label.capitalize())
    ax[i].set_xlabel('Age (5 year bins)')
    sns.move_legend(ax[i], loc='upper left', title=None)
    
ax[0].set_ylabel('Percentage of diseased participants')
fig.suptitle('Age distribution for eye diseases')
plt.tight_layout()

In [ ]:
feature_names = ['basis_uort', 'd_an_aug_1', 'd_an_aug_2', 'd_an_aug_3', 'd_an_metdm2c_1','d_an_met_1']
df = df_labels[feature_names].copy()

feature_name = feature_names[0]
mapping, label = decoder.decode(feature_name)
df, mapping = decoder.group_nans(feature_name, mapping, df)
df[feature_name] = df[feature_name].map(mapping)
df = nako.sort_categorical_df(df, feature_name)

fig, ax = plt.subplots(1, 4, figsize=(18, 4))

for i, feature_name in enumerate(feature_names[2:]):
    mapping, label = decoder.decode(feature_name)
    df, mapping = decoder.group_nans(feature_name, mapping, df)
    df[feature_name] = df[feature_name].map(mapping)

    diseased_by_site = df[['basis_uort', feature_name]].groupby(['basis_uort']).value_counts(normalize=True)

    site, percentage_diseased = [], []
    for item in diseased_by_site.items():
        if item[0][1] == 'Yes':
            site.append(item[0][0])
            percentage_diseased.append(100 * item[1])

    df_diseased_by_site = pd.DataFrame({'Site': site, 'Percentage': percentage_diseased})
    sns.barplot(x='Site', y='Percentage', data=df_diseased_by_site, color=next(colors), ax=ax[i])

    ax[i].set_title(label.capitalize())
    ax[i].set_xlabel('Site')
    ax[i].tick_params(axis='x', rotation=90, labelsize=8)
    
ax[0].set_ylabel('Percentage')
fig.suptitle('Percentage of diseased participants per site')
plt.tight_layout()

### Visual Acuity

In [ ]:
feature_names = ['vi_l_log_mar_einstf', 'vi_r_log_mar_einstf']
df = df_labels[feature_names].copy()

fig, ax = plt.subplots(1, 2, figsize=(7, 3))

for i, feature_name in enumerate(feature_names):
    mapping, label = decoder.decode(feature_name)
    df, mapping = decoder.group_nans(feature_name, mapping, df)
    df[feature_name] = df[feature_name].replace(7.70, np.nan)
    sns.histplot(x=feature_name, data=df, binwidth=0.02, stat='percent', shrink=2, color=next(colors), ax=ax[i])
    ax[i].set_title(label)
    ax[i].set_xlabel('')
    ax[i].set_ylabel('')
    ax[i].set_xticks(np.arange(-0.20, 1.4, 0.2).tolist())
    
ax[0].set_ylabel('Percentage of participants')
plt.tight_layout()

In [ ]:
feature_names = ['basis_sex', 'vi_l_log_mar_einstf', 'vi_r_log_mar_einstf']
df = df_labels[feature_names].copy()

feature_name = feature_names[0]
mapping, label = decoder.decode(feature_name)
df, mapping = decoder.group_nans(feature_name, mapping, df)
df[feature_name] = df[feature_name].map(mapping)
df = helpers.sort_categorical_df(df, feature_name)

fig, ax = plt.subplots(1, 2, figsize=(7, 3))

for i, feature_name in enumerate(feature_names[1:]):
    mapping, label = decoder.decode(feature_name)
    df, mapping = decoder.group_nans(feature_name, mapping, df)
    df[feature_name] = df[feature_name].replace(7.70, np.nan)
    sns.histplot(x=feature_name, data=df, binwidth=0.025, stat='percent', shrink=3, multiple='dodge', color='lightgray', ec=None, ax=ax[i])
    sns.histplot(x=feature_name, data=df, hue='basis_sex', binwidth=0.02, stat='percent', shrink=2, multiple='dodge',ec=None, ax=ax[i])
    sns.move_legend(ax[i], loc='upper right', title=None)
    ax[i].set_title(label)
    ax[i].set_xlabel('')
    ax[i].set_ylabel('')
    ax[i].set_xticks(np.arange(-0.20, 1.4, 0.2).tolist())
    
ax[0].set_ylabel('Percentage of participants')
plt.tight_layout()

In [ ]:
feature_names = ['basis_age', 'basis_sex', 'vi_l_log_mar_einstf', 'vi_r_log_mar_einstf']
df = df_labels[feature_names].copy()

feature_name = feature_names[1]
mapping, label = decoder.decode(feature_name)
df, mapping = decoder.group_nans(feature_name, mapping, df)
df[feature_name] = df[feature_name].map(mapping)
df = utils.sort_categorical_df(df, feature_name)

fig, ax = plt.subplots(1, 2, figsize=(7, 3), sharey=True)

for i, feature_name in enumerate(feature_names[2:]):
    mapping, label = decoder.decode(feature_name)
    df, mapping = decoder.group_nans(feature_name, mapping, df)
    df[feature_name] = df[feature_name].replace(7.70, np.nan)
    sns.scatterplot(data=df, x='basis_age', y=feature_name, alpha=0.8, s=5, marker='.', linewidths=None, edgecolors='None', ax=ax[i])
    # sns.move_legend(ax[i], loc='upper right', title=None)
    ax[i].set_title(label)
    ax[i].set_xlabel('Age')
    # ax[i].set_xticks(np.arange(-0.25, 1.75, 0.25).tolist())
    
ax[0].set_ylabel('logMar')
plt.tight_layout()

In [ ]:
df[['basis_age', 'vi_l_log_mar_einstf', 'vi_r_log_mar_einstf']].corr()

### Neurodegenerative Diseases + Stroke

In [ ]:
feature_names = ['d_an_neu_1', 'd_an_neu_4', 'd_an_sel_8']
df = df_labels[feature_names].copy()

fig, ax = plt.subplots(1, 3, figsize=(12, 4))

for i, feature_name in enumerate(feature_names):
    mapping, label = decoder.decode(feature_name)
    df, mapping = decoder.group_nans(feature_name, mapping, df)
    df[feature_name] = df[feature_name].map(mapping)
    df = nako.sort_categorical_df(df, feature_name) 
    sns.histplot(x=feature_name, data=df, stat='percent', shrink=0.7, color=next(colors), ax=ax[i])
    ax[i].set_title(label.capitalize())
    ax[i].bar_label(ax[i].containers[0], fmt='%.3f')
    ax[i].set_xlabel('')
    ax[i].set_ylabel('')
    
ax[0].set_ylabel('Percentage of participants')
plt.tight_layout()

In [ ]:
feature_names = ['basis_age', 'd_an_neu_1', 'd_an_neu_4', 'd_an_sel_8']
df = df_labels[feature_names].copy()

fig, ax = plt.subplots(1, 3, figsize=(12, 4))

for i, feature_name in enumerate(feature_names[1:]):
    mapping, label = decoder.decode(feature_name)
    df, mapping = decoder.group_nans(feature_name, mapping, df)
    df[feature_name] = df[feature_name].map(mapping)

    df_filtered = df[df[feature_name] == 'Yes']
    sns.histplot(x='basis_age', data=df_filtered, bins=age_bins, stat='percent', shrink=0.7, color=next(colors), ax=ax[i])

    ax[i].set_xticks(list(range(20, 80, 10)))
    ax[i].set_title(label.capitalize())
    ax[i].set_xlabel('Age (5 year bins)')
    ax[i].set_ylabel('')
    
ax[0].set_ylabel('Percentage of diseased participants')
fig.suptitle('Age distribution for eye diseases')
plt.tight_layout()

In [ ]:
feature_names = ['d_an_neu_1', 'd_an_neu_4', 'd_an_sel_8']
df = df_labels[feature_names].copy()

for i, feature_name in enumerate(feature_names):
    mapping, label = decoder.decode(feature_name)
    df, mapping = decoder.group_nans(feature_name, mapping, df)
    count = df[feature_name].map(mapping).value_counts(normalize=False)
    count = [f'{index}: {value}' for index, value in count.items()]
    
    print(label.capitalize())
    print(', '.join(count))